# Finding new clusters for better classifying

Ticket link: https://github.com/kowaalczyk/astronomical-classification/projects/1#card-15537673a

In [ ]:
import gc
import pandas as pd
import numpy as np
import dask.dataframe as dd

from datetime import datetime as dt
from functools import partial, wraps

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from numba import jit
import matplotlib.pyplot as plt
from experiments import start_run, log_args
import plasticc.xgb_train as xgb
import plasticc.lgbm_train as lgbm
from plasticc.training import process_meta
from plasticc.featurize import featurize
from plasticc.lgbm_train import lgbm_modeling_cross_validation
from plasticc.xgb_train import xgb_modeling_cross_validation

In [ ]:
gc.enable()

In [ ]:
training_df = dd.read_csv("../../../kk385830/astronomical-classification/data/raw/training_set.csv")
training_meta_df = dd.read_csv("../../../kk385830/astronomical-classification/data/raw/training_set_metadata.csv")
test_df = dd.read_csv("../../../kk385830/astronomical-classification/data/raw/test_set.csv")
test_meta_df = dd.read_csv("../../../kk385830/astronomical-classification/data/raw/test_set_metadata.csv")

## Data featurization
We will now featurize the training data and using CV we will check how does it perform

In [ ]:
fcp = {
    'flux': {
        'longest_strike_above_mean': None,
        'longest_strike_below_mean': None,
        'mean_change': None,
        'mean_abs_change': None,
        'length': None,
    },

    'flux_by_flux_ratio_sq': {
        'longest_strike_above_mean': None,
        'longest_strike_below_mean': None,       
    },

    'flux_passband': {
        'fft_coefficient': [
                {'coeff': 0, 'attr': 'abs'}, 
                {'coeff': 1, 'attr': 'abs'}
            ],
        'kurtosis' : None, 
        'skewness' : None,
    },

    'mjd': {
        'maximum': None, 
        'minimum': None,
        'mean_change': None,
        'mean_abs_change': None,
    },
}

In [ ]:
aggs = {
    'flux': ['min', 'max', 'mean', 'median', 'std', 'skew'],
    'flux_err': ['min', 'max', 'mean', 'median', 'std', 'skew'],
    'detected': ['mean'],
    'flux_ratio_sq':['sum', 'skew'],
    'flux_by_flux_ratio_sq':['sum','skew'],
}

In [ ]:
%%time
meta_train = process_meta("../../../kk385830/astronomical-classification/data/raw/training_set_metadata.csv")
train = pd.read_csv("../../../kk385830/astronomical-classification/data/raw/training_set.csv")

In [ ]:
%%time
X = featurize(train, meta_train, aggs, fcp)
X_backup = X.copy()

## Setting up CV

In [ ]:
# lgbm_params = {
#     'device': 'cpu',
#     'objective': 'multiclass',
#     'num_class': 14,
#     'boosting_type': 'gbdt',
#     'n_jobs': 16,
#     'max_depth': 7,
#     'n_estimators': 1024,
#     'subsample_freq': 2,
#     'subsample_for_bin': 5000,
#     'min_data_per_group': 100,
#     'max_cat_to_onehot': 4,
#     'cat_l2': 1.0,
#     'cat_smooth': 59.5,
#     'max_cat_threshold': 32,
#     'metric_freq': 10,
#     'verbosity': -1,
#     'metric': 'multi_logloss',
#     'xgboost_dart_mode': False,
#     'uniform_drop': False,
#     'colsample_bytree': 0.5,
#     'drop_rate': 0.173,
#     'learning_rate': 0.0267,
#     'max_drop': 5,
#     'min_child_samples': 10,
#     'min_child_weight': 100.0,
#     'min_split_gain': 0.1,
#     'num_leaves': 7,
#     'reg_alpha': 0.1,
#     'reg_lambda': 0.00023,
#     'skip_drop': 0.44,
#     'subsample': 0.75
# }
lgbm_params = {
    'device': 'cpu',
    'objective': 'multiclass',
    'num_class': 14,
    'boosting_type': 'gbdt',
    'n_jobs': 16,
    'max_depth': 7,
    'n_estimators': 1024,
    'subsample_freq': 2,
    'subsample_for_bin': 5000,
    'min_data_per_group': 100,
    'max_cat_to_onehot': 4,
    'cat_l2': 1.0,
    'cat_smooth': 59.5,
    'max_cat_threshold': 32,
    'metric_freq': 10,
    'verbosity': -1,
    'metric': 'multi_logloss',
    'xgboost_dart_mode': False,
    'uniform_drop': False,
    'colsample_bytree': 0.5,
    'drop_rate': 0.173,
    'learning_rate': 0.0267,
    'max_drop': 5,
    'min_child_samples': 10,
    'min_child_weight': 100.0,
    'min_split_gain': 0.1,
    'num_leaves': 7,
    'reg_alpha': 0.1,
    'reg_lambda': 0.00023,
    'skip_drop': 0.44,
    'subsample': 0.75,
    'max_bin': 127
}

In [ ]:
if 'target' in X:
    y = X['target']
    del X['target']
else:
    print("What the heck")

In [ ]:
# Taken from Giba's topic : https://www.kaggle.com/titericz
# https://www.kaggle.com/c/PLAsTiCC-2018/discussion/67194
# with Kyle Boone's post https://www.kaggle.com/kyleboone
classes = sorted(y.unique())
class_weights = {c: 1 for c in classes}
class_weights.update({c:2 for c in [64, 15]})
print('Unique classes : {}, {}'.format(len(classes), classes))
print(class_weights)

In [ ]:
def test_partition(mask, lgbm_conf1, lgbm_conf2):
    """ mask is a lambda operating on training set, returning boolean values to split the set"""
    result_mine = []
    X_after1 = X_backup[mask]
    X_after2 = X_backup[~mask]
    y1 = 0
    y2 = 0
    for t in [[X_after1, y1, lgbm_conf1], [X_after2, y2, lgbm_conf2]]:
        if 'object_id' in t[0]:
            t[1] = pd.Series(t[0]['target'])
            del t[0]['target']
            del t[0]['object_id'] 
            del t[0]['hostgal_specz']
            del t[0]['ra'], t[0]['decl'], t[0]['gal_l'], t[0]['gal_b']
            del t[0]['ddf']
        else:
            print(f"What the heck: {t}")
        eval_func = partial(
            lgbm_modeling_cross_validation, 
            X=t[0], 
            y=t[1], 
            classes=classes, 
            class_weights=class_weights, 
            nr_fold=5, 
            random_state=1
        )
        clfs, score = eval_func(t[2])
        result_mine.append(score)
    print(result_mine)

### Baseline
Before starting, let's take a look at how lgbm perform on full training set

In [ ]:
lgbm_params = {
    'device': 'cpu',
    'objective': 'multiclass',
    'num_class': 14,
    'boosting_type': 'gbdt',
    'n_jobs': -1,
    'max_depth': 6,
    'n_estimators': 1024,
    'subsample_freq': 2,
    'subsample_for_bin': 5000,
    'min_data_per_group': 100,
    'max_cat_to_onehot': 4,
    'cat_l2': 1.0,
    'cat_smooth': 59.5,
    'max_cat_threshold': 32,
    'metric_freq': 10,
    'verbosity': -1,
    'metric': 'multi_logloss',
    'xgboost_dart_mode': False,
    'uniform_drop': False,
    'colsample_bytree': 0.5,
    'drop_rate': 0.173,
    'learning_rate': 0.0267,
    'max_drop': 5,
    'min_child_samples': 10,
    'min_child_weight': 100.0,
    'min_split_gain': 0.125,
    'num_leaves': 7,
    'reg_alpha': 0.1,
    'reg_lambda': 0.00023,
    'skip_drop': 0.44,
    'subsample': 0.75,
    'max_bin': 127,
    'min_data_in_leaf': 15,
    'lambda_l1': 2,
}
baseline_X = X_backup.copy()
if 'object_id' in baseline_X:
    baseline_Y = pd.Series(baseline_X['target'])
    del baseline_X['target']
    del baseline_X['object_id'] 
    del baseline_X['hostgal_specz']
    del baseline_X['ra'], baseline_X['decl'], baseline_X['gal_l'], baseline_X['gal_b']
    del baseline_X['ddf']
else:
    print("What the heck")
eval_func = partial(
    lgbm_modeling_cross_validation, 
    X=baseline_X, 
    y=baseline_Y, 
    classes=classes, 
    class_weights=class_weights, 
    nr_fold=5, 
    random_state=1
)
clfs, score = eval_func(lgbm_params)
print(score)

Score: 0.6501199352977585  
0.6463362734878486 'max_bin': 95,  
0.6461180983317429 'max_bin': 127,  
0.6450608843517845 'max_bin': 127, 'min_data_in_leaf': 15  
0.6445799942947439 'max_bin': 127, 'min_data_in_leaf': 15, 'lambda_l1': 2,  
0.6442824787482899 'max_bin': 127, 'min_data_in_leaf': 15, 'lambda_l1': 2, 'min_split_gain': 0.125,


### First Idea:
Let's split data by first feature from metadata - ra(right ascension, sky coordinate: co-longitude in degrees).  
We split somewhere between 250 and 300 -  it looks quite promising as both of the sets have similar distribution

In [ ]:
plt.hist(training_meta_df.ra.compute(),  bins=range(0, 370))

In [ ]:
plt.hist(test_meta_df.ra.compute(), bins=range(0, 370))

With scores around 0.7031538683880986, 0.8434040355315483 for the first and second subset, I find it pointless to continue exploration

In [ ]:
#[0.7031538683880986, 0.8434040355315483] default
conf1 = lgbm_params.copy()
conf2 = lgbm_params.copy()
test_partition(X_backup.ra < 300, conf1, conf2)

### Second Idea:
Now it's time for gal_l, but spliting into to sets where one of them a few times smaller than the second one wouldn't make sense.  
[1.023404000997548, 0.6577501966844485]

In [ ]:
plt.hist(test_meta_df.gal_l.compute())

In [ ]:
plt.hist(training_meta_df.gal_l.compute())

In [ ]:
#[0.7031538683880986, 0.8434040355315483] default
conf1 = lgbm_params.copy()
conf2 = lgbm_params.copy()
test_partition(X_backup.gal_l < 125, conf1, conf2)

### Third Idea:
I think look at test data histogram shows firmly, that this wouldn't make sense

In [ ]:
plt.hist(training_meta_df.ddf.compute())

In [ ]:
plt.hist(test_meta_df.ddf.compute())

### Fourth Idea:  
No space for splitting set, I guess

In [ ]:
plt.hist(training_meta_df.distmod.compute())
print(training_meta_df.shape[0].compute())
print(training_meta_df.distmod.isna().sum().compute())

In [ ]:
plt.hist(test_meta_df.distmod.compute())
print(test_meta_df.shape[0].compute())
print(test_meta_df.distmod.isna().sum().compute())

### Fifth Idea:  
Unfortunately pointless (I was hoping for it)

In [ ]:
plt.hist(training_df.groupby(['object_id']).mean().compute().detected)

### Sixth Idea:  
Also pointless

In [ ]:
plt.hist(training_df.groupby(['object_id']).mean().compute().flux_err, bins=range(0, 100))

In [ ]:
plt.hist(test_df.groupby(['object_id']).mean().compute().flux_err, bins=range(0, 100))

### Seventh Idea:  
Also pointless

In [ ]:
plt.hist(training_df.groupby(['object_id']).count().compute().flux)

In [ ]:
plt.hist(test_df.groupby(['object_id']).count().compute().flux)

### Eighth Idea:  
Also pointless

In [ ]:
plt.hist(training_df.groupby(['object_id', 'passband']).count().compute().flux)

In [ ]:
plt.hist(test_df[0:10000].groupby(['object_id', 'passband']).count().compute().flux)

### Ninth Idea:  
Also pointless

In [ ]:
plt.hist(training_df.groupby(['object_id', 'passband']).max().compute().mjd - training_df.groupby(['object_id', 'passband']).min().compute().mjd)

In [ ]:
plt.hist(test_df[0:4000].groupby(['object_id', 'passband']).max().compute().mjd - test_df[0:4000].groupby(['object_id', 'passband']).min().compute().mjd)